In [1]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score

import nltk

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import seaborn as sns

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("saurabhshahane/fake-news-classification")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fake-news-classification' dataset.
Path to dataset files: /kaggle/input/fake-news-classification


In [3]:
df = pd.read_csv(path + "/WELFake_Dataset.csv")
df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [4]:
def pipeline():
  # Combine 'title' and 'text' columns into a single 'review' column
  # Fill NaN values with empty strings before combining to avoid errors
  df['full_review'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
  review_text = df['full_review'].iloc[0] # Get the first combined review as a string

  # tokenization
  tokenized_review = word_tokenize(review_text)
  print(tokenized_review)

  # lemmatization
  lemmatizer = WordNetLemmatizer()
  lemmatized_review = [lemmatizer.lemmatize(word) for word in tokenized_review]
  print(lemmatized_review)

  # stopword
  stop_words = set(stopwords.words('english'))
  filter_review = [word for word in tokenized_review if not word.lower() in stop_words]
  print(filter_review)


In [6]:
if __name__ == "__main__":
  pipeline()

['LAW', 'ENFORCEMENT', 'ON', 'HIGH', 'ALERT', 'Following', 'Threats', 'Against', 'Cops', 'And', 'Whites', 'On', '9-11By', '#', 'BlackLivesMatter', 'And', '#', 'FYF911', 'Terrorists', '[', 'VIDEO', ']', 'No', 'comment', 'is', 'expected', 'from', 'Barack', 'Obama', 'Members', 'of', 'the', '#', 'FYF911', 'or', '#', 'FukYoFlag', 'and', '#', 'BlackLivesMatter', 'movements', 'called', 'for', 'the', 'lynching', 'and', 'hanging', 'of', 'white', 'people', 'and', 'cops', '.', 'They', 'encouraged', 'others', 'on', 'a', 'radio', 'show', 'Tuesday', 'night', 'to', 'turn', 'the', 'tide', 'and', 'kill', 'white', 'people', 'and', 'cops', 'to', 'send', 'a', 'message', 'about', 'the', 'killing', 'of', 'black', 'people', 'in', 'America.One', 'of', 'the', 'F', '*', '*', '*', 'YoFlag', 'organizers', 'is', 'called', 'Sunshine', '.', 'She', 'has', 'a', 'radio', 'blog', 'show', 'hosted', 'from', 'Texas', 'called', ',', 'Sunshine', 's', 'F', '*', '*', '*', 'ing', 'Opinion', 'Radio', 'Show', '.', 'A', 'snapshot'

In [18]:
df.head()

,Unnamed: 0,title,text,label,full_review
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1,LAW ENFORCEMENT ON HIGH ALERT Following Threat...
1,1,NaN,Did they post their votes for Hillary already?,1,Did they post their votes for Hillary already?
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0,"Bobby Jindal, raised Hindu, uses story of Chri..."
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1,SATAN 2: Russia unvelis an image of its terrif...


In [7]:
x = df['full_review']
y = df['label']

In [8]:
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

In [9]:
# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # You can adjust max_features as needed

# Fit and transform the training data
X_train_tfidf = tfidf_vectorizer.fit_transform(x_train)

# Transform the test data
X_test_tfidf = tfidf_vectorizer.transform(x_test)

print("Shape of TF-IDF vectorized training data:", X_train_tfidf.shape)
print("Shape of TF-IDF vectorized test data:", X_test_tfidf.shape)

Shape of TF-IDF vectorized training data: (57707, 5000)
Shape of TF-IDF vectorized test data: (14427, 5000)


In [10]:
# Initialize and train a Logistic Regression model
log_reg_model = LogisticRegression(max_iter=1000) # Increase max_iter for convergence
log_reg_model.fit(X_train_tfidf, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [11]:
y_pred = log_reg_model.predict(X_test_tfidf)
print("__PREDICTION_COMPLETE__")

__PREDICTION_COMPLETE__


In [12]:
# confusion matrix
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(cm)

# accuracy
acc = accuracy_score(y_test, y_pred)
print('Accuracy:', acc)

# precision
precision = precision_score(y_test, y_pred, pos_label=0)
print('Precision:', precision)

# recall
recall = recall_score(y_test, y_pred, pos_label=0)
print('Recall:', recall)

# f1_score
f1 = f1_score(y_test, y_pred, pos_label=0)
print('F1 Score:', f1)

# classification report
print('Classification Report:')
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[6675  414]
 [ 337 7001]]
Accuracy: 0.9479448256740833
Precision: 0.951939532230462
Recall: 0.9415996614473128
F1 Score: 0.9467413658605772
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.94      0.95      7089
           1       0.94      0.95      0.95      7338

    accuracy                           0.95     14427
   macro avg       0.95      0.95      0.95     14427
weighted avg       0.95      0.95      0.95     14427



In [13]:
vectorizer = CountVectorizer()
x_v = vectorizer.fit_transform(x_test)
y_v = df['label']
vectorizer.get_feature_names_out()

array(['00', '000', '0000', ..., 'کنیم', 'ᱚɛɱ', 'めで鯛'], dtype=object)

In [14]:
x_v.toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0]])

In [17]:
text = input("Enter a review: ")
vec = tfidf_vectorizer.transform([text])
results = log_reg_model.predict(vec)
print(results)
if results == 1:
  print("Fake News")
else:
  print("Real News")

Enter a review: PM of UK killed a person on the streets of Lucknow last night.
[1]
Fake News


# Task
Generate a report on fake news classification using the `WELFake_Dataset.csv` from the `saurabhshahane/fake-news-classification` dataset, including:
*   An overview of the project and dataset.
*   Details on text preprocessing (tokenization, lemmatization, stop word removal) and TF-IDF vectorization (with its mathematical formula).
*   A description of the Logistic Regression model used.
*   Definitions and mathematical formulas for Accuracy, Precision, Recall, F1-Score, and Confusion Matrix, along with the model's performance metrics.
*   A conclusion summarizing findings and suggesting future improvements.

## Project Overview

### Subtask:
Provide a brief introduction to the project, its objective (fake news classification), and the dataset used.


## Project Overview

### Subtask:
Provide a brief introduction to the project, its objective (fake news classification), and the dataset used.

### Instructions
1. Write a text cell that introduces the project, explaining its main goal: classifying fake news.
2. In the same text cell, describe the dataset used, specifically mentioning that it's the `WELFake_Dataset.csv` obtained from the `saurabhshahane/fake-news-classification` dataset.
3. Briefly explain why this project is important.

***

### Introduction to the Project
This project aims to tackle the pervasive problem of fake news by developing a classification model capable of distinguishing between real and fake news articles. The primary objective is to build a robust system that can accurately identify misleading or fabricated information, thereby helping to combat misinformation and promote media literacy.

### Dataset Used
The dataset utilized for this project is `WELFake_Dataset.csv`, which was obtained from the `saurabhshahane/fake-news-classification` dataset on Kaggle. This dataset contains a collection of news articles, each labeled as either 'real' or 'fake', and includes features such as the article's title and text, which are crucial for training our classification model.

### Importance of the Project
The proliferation of fake news has significant societal implications, ranging from influencing public opinion and political processes to eroding trust in established media outlets. By accurately classifying fake news, this project contributes to creating a more informed public, safeguarding democratic processes, and promoting a healthier information ecosystem.

## Data Preprocessing

### Subtask:
Describe the text preprocessing steps (tokenization, lemmatization, stop word removal) and explain the TF-IDF vectorization process, including its mathematical formula.


## Data Preprocessing

### Subtask:
Describe the text preprocessing steps (tokenization, lemmatization, stop word removal) and explain the TF-IDF vectorization process, including its mathematical formula.

#### Instructions

### Text Preprocessing Steps:

In this notebook, the following text preprocessing steps were performed within the `pipeline()` function to prepare the text data for machine learning:

1.  **Tokenization**: This is the process of breaking down raw text into smaller units called tokens. Each token typically represents a word or a punctuation mark. The purpose of tokenization is to convert the continuous text into discrete units that can be processed and analyzed. In our case, `nltk.tokenize.word_tokenize` was used to split the `full_review` column into individual words.

2.  **Lemmatization**: This is the process of reducing inflected words to their base or root form, known as a lemma. Unlike stemming, which often chops off word endings, lemmatization uses vocabulary and morphological analysis of words, aiming to return the dictionary form of a word. For example, 'running', 'runs', and 'ran' would all be lemmatized to 'run'. This helps in reducing the dimensionality of the data and ensures that different forms of the same word are treated as a single entity. `nltk.stem.WordNetLemmatizer` was used for this purpose.

3.  **Stop Word Removal**: Stop words are common words (e.g., 'the', 'is', 'a', 'an', 'in') that often carry little significant meaning and can be removed without altering the overall sentiment or context of a sentence. Removing them helps to reduce noise in the data, decrease processing time, and improve the performance of text classification models by focusing on more informative words. `nltk.corpus.stopwords` was utilized to filter out these common words.

### TF-IDF (Term Frequency-Inverse Document Frequency) Vectorization:

TF-IDF is a numerical statistic that is intended to reflect how important a word is to a document in a collection or corpus. It is commonly used as a weighting factor in information retrieval and text mining. The intuition behind TF-IDF is that words that appear frequently in a document but rarely across the entire corpus are more likely to be relevant to that specific document.

**How it works:**

*   **Term Frequency (TF)**: Measures how frequently a term appears in a document. Since every document is different in length, it is possible that a term would appear more times in longer documents than shorter ones. Thus, the term frequency is often divided by the document length (or total number of terms in the document) as a normalization step.

*   **Inverse Document Frequency (IDF)**: Measures how important a term is across the whole corpus. When computing IDF, it is important to note that rare terms that appear in only a few documents have a high IDF score, whereas common terms that appear in many documents have a low IDF score.

**Mathematical Formula for TF-IDF:**

The TF-IDF score for a term \(t\) in a document \(d\) within a corpus \(D\) is calculated as:

\[\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)\]

Where:

*   **Term Frequency (TF):**
    \[\text{TF}(t, d) = \frac{\text{Number of times term } t \text{ appears in document } d}{\text{Total number of terms in document } d}\]

*   **Inverse Document Frequency (IDF):**
    \[\text{IDF}(t, D) = \log_{e}\left(\frac{\text{Total number of documents } D}{\text{Number of documents with term } t \text{ in them}}\right)\]

In this notebook, `TfidfVectorizer` from `sklearn.feature_extraction.text` was used to implement TF-IDF. Specifically, the `max_features` parameter was set to `5000`, meaning that only the 5000 most frequently occurring terms with the highest TF-IDF scores were considered for vectorization, reducing the dimensionality of the feature space.

## Model Description

### Subtask:
Detail the Logistic Regression model used for classification.


## Model Description

### Subtask:
Detail the Logistic Regression model used for classification.

### Logistic Regression Overview
Logistic Regression is a statistical model used for binary classification tasks, meaning it predicts a binary outcome (e.g., true/false, 0/1). Despite its name, it is a classification algorithm rather than a regression algorithm. It works by estimating the probability that a given instance belongs to a particular class. If the probability is greater than a certain threshold (usually 0.5), it assigns the instance to that class; otherwise, it assigns it to the other class. The model uses a sigmoid function to map predicted values to probabilities between 0 and 1.

### Application in this Project
In this project, Logistic Regression was employed to classify news articles as either 'fake' (1) or 'real' (0). Before feeding the text data to the model, the 'full_review' column (which combines 'title' and 'text') was transformed into numerical features using the **TF-IDF Vectorizer**. TF-IDF (Term Frequency-Inverse Document Frequency) converts text into numerical vectors, where each value reflects the importance of a word in a document relative to its frequency in the entire corpus. This numerical representation was then used as input (`X_train_tfidf` and `X_test_tfidf`) for the Logistic Regression model.

### Key Parameters
The `LogisticRegression` model was initialized with the following important parameter:
- `max_iter=1000`: This parameter sets the maximum number of iterations for the solver to converge. Logistic Regression uses iterative optimization algorithms to find the best-fitting coefficients. If the algorithm does not converge within the default number of iterations, increasing `max_iter` allows it more chances to find a solution. In this case, it was increased to `1000` to ensure convergence, especially with a large dataset and a high-dimensional feature space from TF-IDF.

## Evaluation Metrics

### Subtask:
Define and provide the mathematical formulas for the key evaluation metrics: Accuracy, Precision, Recall, F1-Score, and Confusion Matrix. Present the results obtained from the model.


## Evaluation Metrics

### Confusion Matrix
A Confusion Matrix is a table used to describe the performance of a classification model on a set of test data for which the true values are known. It allows visualization of the performance of an algorithm. Each row of the matrix represents the instances in an actual class, while each column represents the instances in a predicted class.

In the context of fake news classification:
*   **True Positive (TP)**: The model correctly predicted news as fake when it was actually fake.
*   **True Negative (TN)**: The model correctly predicted news as real when it was actually real.
*   **False Positive (FP)**: The model incorrectly predicted news as fake when it was actually real (Type I error).
*   **False Negative (FN)**: The model incorrectly predicted news as real when it was actually fake (Type II error).

Our model's Confusion Matrix:
```
[[6675  414]
 [ 337 7001]]
```
From the matrix:
*   TP (Predicted 1, Actual 1): 7001
*   TN (Predicted 0, Actual 0): 6675
*   FP (Predicted 1, Actual 0): 414
*   FN (Predicted 0, Actual 1): 337

### Accuracy
Accuracy measures the proportion of total predictions that were correct. It's a good general measure, but can be misleading in imbalanced datasets.

**Formula:**
$Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$

**Our model's Accuracy:** 0.9479

### Precision
Precision measures the proportion of positive identifications that were actually correct. It answers the question: "Of all the instances predicted as positive, how many were actually positive?"

**Formula (for class 0 - Real News):**
$Precision = \frac{TN}{TN + FN}$

**Our model's Precision (for class 0):** 0.9519

### Recall
Recall (also known as Sensitivity) measures the proportion of actual positives that were correctly identified. It answers the question: "Of all the actual positive instances, how many did we correctly predict?"

**Formula (for class 0 - Real News):**
$Recall = \frac{TN}{TN + FP}$

**Our model's Recall (for class 0):** 0.9416

### F1-Score
The F1-Score is the harmonic mean of Precision and Recall. It provides a single score that balances both concerns and is particularly useful when you need to balance Precision and Recall, especially in cases of uneven class distribution.

**Formula:**
$F1-Score = 2 \times \frac{Precision \times Recall}{Precision + Recall}$

**Our model's F1-Score (for class 0):** 0.9467

### Classification Report
The classification report provides a more detailed breakdown of these metrics for each class, along with support (the number of actual occurrences of the class).

```
              precision    recall  f1-score   support

           0       0.95      0.94      0.95      7089
           1       0.94      0.95      0.95      7338

    accuracy                           0.95     14427
   macro avg       0.95      0.95      0.95     14427
weighted avg       0.95      0.95      0.95     14427
```

## Evaluation Metrics

### Confusion Matrix
A Confusion Matrix is a table used to describe the performance of a classification model on a set of test data for which the true values are known. It allows visualization of the performance of an algorithm. Each row of the matrix represents the instances in an actual class, while each column represents the instances in a predicted class.

In the context of fake news classification:
*   **True Positive (TP)**: The model correctly predicted news as fake when it was actually fake.
*   **True Negative (TN)**: The model correctly predicted news as real when it was actually real.
*   **False Positive (FP)**: The model incorrectly predicted news as fake when it was actually real (Type I error).
*   **False Negative (FN)**: The model incorrectly predicted news as real when it was actually fake (Type II error).

Our model's Confusion Matrix:
```
[[6675  414]
 [ 337 7001]]
```
From the matrix:
*   TP (Predicted 1, Actual 1): 7001
*   TN (Predicted 0, Actual 0): 6675
*   FP (Predicted 1, Actual 0): 414
*   FN (Predicted 0, Actual 1): 337

### Accuracy
Accuracy measures the proportion of total predictions that were correct. It's a good general measure, but can be misleading in imbalanced datasets.

**Formula:**
$Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$

**Our model's Accuracy:** 0.9479

### Precision
Precision measures the proportion of positive identifications that were actually correct. It answers the question: "Of all the instances predicted as positive, how many were actually positive?"

**Formula (for class 0 - Real News):**
$Precision = \frac{TN}{TN + FN}$

**Our model's Precision (for class 0):** 0.9519

### Recall
Recall (also known as Sensitivity) measures the proportion of actual positives that were correctly identified. It answers the question: "Of all the actual positive instances, how many did we correctly predict?"

**Formula (for class 0 - Real News):**
$Recall = \frac{TN}{TN + FP}$

**Our model's Recall (for class 0):** 0.9416

### F1-Score
The F1-Score is the harmonic mean of Precision and Recall. It provides a single score that balances both concerns and is particularly useful when you need to balance Precision and Recall, especially in cases of uneven class distribution.

**Formula:**
$F1-Score = 2 \times \frac{Precision \times Recall}{Precision + Recall}$

**Our model's F1-Score (for class 0):** 0.9467

### Classification Report
The classification report provides a more detailed breakdown of these metrics for each class, along with support (the number of actual occurrences of the class).

```
              precision    recall  f1-score   support

           0       0.95      0.94      0.95      7089
           1       0.94      0.95      0.95      7338

    accuracy                           0.95     14427
   macro avg       0.95      0.95      0.95     14427
weighted avg       0.95      0.95      0.95     14427
```

## Conclusion

### Subtask:
Summarize the project findings and discuss potential future improvements.


## Summary:

### Q&A
The project successfully developed a classification model to distinguish between real and fake news articles.

### Data Analysis Key Findings
*   The project used the `WELFake_Dataset.csv` from the `saurabhshahane/fake-news-classification` dataset, which contains news articles labeled as 'real' or 'fake'.
*   Text preprocessing involved tokenization using `nltk.tokenize.word_tokenize`, lemmatization using `nltk.stem.WordNetLemmatizer`, and stop word removal using `nltk.corpus.stopwords`.
*   TF-IDF vectorization was applied using `TfidfVectorizer` with `max_features` set to `5000` to convert text into numerical features, representing term importance in documents.
*   A Logistic Regression model was employed for classification, configured with `max_iter=1000` to ensure convergence.
*   The model achieved a high overall accuracy of approximately 94.79%.
*   The confusion matrix showed:
    *   True Positives (correctly predicted fake): 7001
    *   True Negatives (correctly predicted real): 6675
    *   False Positives (incorrectly predicted fake): 414
    *   False Negatives (incorrectly predicted real): 337
*   Detailed performance metrics from the classification report indicate strong performance for both classes:
    *   For class 0 (real news): Precision was 0.95, Recall was 0.94, and F1-Score was 0.95.
    *   For class 1 (fake news): Precision was 0.94, Recall was 0.95, and F1-Score was 0.95.

### Insights or Next Steps
*   The Logistic Regression model demonstrates strong performance in fake news classification, with high precision and recall for both real and fake news, indicating its effectiveness in combating misinformation.
*   Future improvements could explore more advanced natural language processing techniques (e.g., word embeddings, deep learning models) and experiment with different feature engineering approaches to potentially enhance model robustness and generalization.
